# Initial Exploration for Shortest Function Finder

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# This is a placeholder for future explorations.
print('Hello from the notebook!')

# Example: Define a simple 1D function and generate some data
def target_function(x):
    return np.sin(x * 2) + x * 0.5

x_data = np.linspace(-5, 5, 100)
y_data = target_function(x_data) + np.random.normal(0, 0.2, x_data.shape)

plt.figure(figsize=(10, 6))
plt.scatter(x_data, y_data, label='Sample Data', color='blue', s=10)
plt.plot(x_data, target_function(x_data), label='True Function', color='red', linestyle='--')
plt.title('Sample 1D Data for Exploration')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Symbolic Regression with gplearn
from gplearn.genetic import SymbolicRegressor
from sklearn.metrics import mean_squared_error

# Reshape x_data for gplearn (it expects 2D array for features)
X_train_gp = x_data.reshape(-1, 1)
y_train_gp = y_data

# Initialize SymbolicRegressor
# These are basic parameters; they can be tuned for better performance or faster convergence
est_gp = SymbolicRegressor(population_size=1000, # Reduced for quicker execution in demo
                           generations=20,       # Reduced for quicker execution in demo
                           stopping_criteria=0.01,
                           p_crossover=0.7,
                           p_subtree_mutation=0.1,
                           p_hoist_mutation=0.05,
                           p_point_mutation=0.1,
                           max_samples=0.9,
                           verbose=1,
                           function_set=('add', 'sub', 'mul', 'div', 'sin', 'cos', 'log', 'sqrt', 'neg'), # Added more functions
                           metric='mean absolute error', # More robust to outliers than mse
                           parsimony_coefficient=0.005, # Penalize overly complex functions
                           random_state=0,
                           n_jobs=-1) # Use all available cores

print("Starting Symbolic Regression fitting...")
est_gp.fit(X_train_gp, y_train_gp)
print("Symbolic Regression fitting complete.")

print(f"Best symbolic function: {est_gp._program}")

# Predict using the gplearn model
y_pred_gp = est_gp.predict(X_train_gp)

# Plot the results
plt.figure(figsize=(12, 7))
plt.scatter(x_data, y_data, label='Sample Data', color='blue', s=10, alpha=0.7)
plt.plot(x_data, target_function(x_data), label='True Function', color='red', linestyle='--')
plt.plot(x_data, y_pred_gp, label=f'gplearn Prediction: {est_gp._program}', color='green', linestyle='-.')
plt.title('Symbolic Regression with gplearn')
plt.xlabel('x')
plt.ylabel('y')
plt.legend(loc='best', fontsize='small') # Adjusted legend location and font size
plt.grid(True)
plt.show()

# Calculate Mean Squared Error for the gplearn model
mse_gp = mean_squared_error(y_train_gp, y_pred_gp)
print(f"Mean Squared Error (gplearn): {mse_gp}")

# Display the program string with better formatting if possible
# (Note: est_gp._program is already the string representation)
print("\nDetails of the best program:")
print(f"  Function: {est_gp._program}")
print(f"  Length: {est_gp._program.length_}")
print(f"  Fitness (MAE): {est_gp._program.raw_fitness_}")
print(f"  Parsimony coefficient: {est_gp.parsimony_coefficient}")


## Experimenting with gplearn Parameters

Let's try to tune some of the `SymbolicRegressor` parameters to see their effect on the resulting function. We'll focus on `population_size`, `generations`, and `parsimony_coefficient`.

In [ ]:
# Re-use data from previous cell
# X_train_gp = x_data.reshape(-1, 1) # This is already available from the previous cell
# y_train_gp = y_data # This is also available

# Define parameter sets to experiment with
param_sets = [
    {
        'name': 'Baseline (Quick Run)',
        'population_size': 500, 'generations': 15, 'parsimony_coefficient': 0.005,
        'stopping_criteria': 0.01, 'verbose': 1 
    },
    {
        'name': 'Larger Population & Generations',
        'population_size': 2000, 'generations': 30, 'parsimony_coefficient': 0.005,
        'stopping_criteria': 0.01, 'verbose': 1
    },
    {
        'name': 'Higher Parsimony (Simpler Functions)',
        'population_size': 1000, 'generations': 20, 'parsimony_coefficient': 0.05, # Increased parsimony
        'stopping_criteria': 0.01, 'verbose': 1
    },
    {
        'name': 'Lower Parsimony (Potentially More Complex)',
        'population_size': 1000, 'generations': 20, 'parsimony_coefficient': 0.0005, # Decreased parsimony
        'stopping_criteria': 0.01, 'verbose': 1
    }
]

common_params = {
    'function_set': ('add', 'sub', 'mul', 'div', 'sin', 'cos', 'log', 'sqrt', 'neg'),
    'metric': 'mean absolute error',
    'p_crossover': 0.7, 'p_subtree_mutation': 0.1, 'p_hoist_mutation': 0.05, 'p_point_mutation': 0.1,
    'max_samples': 0.9, 'random_state': 42, 'n_jobs': -1
}

results = []

for params_dict in param_sets:
    run_name = params_dict.pop('name')
    current_params = {**common_params, **params_dict}
    
    print(f"\n--- Running Experiment: {run_name} ---")
    print(f"Parameters: {params_dict}") # Print only the varied params for brevity
    
    est_gp_tuned = SymbolicRegressor(**current_params)
    
    print("Fitting regressor...")
    est_gp_tuned.fit(X_train_gp, y_train_gp)
    print("Fitting complete.")
    
    program_str = str(est_gp_tuned._program)
    mse = mean_squared_error(y_train_gp, est_gp_tuned.predict(X_train_gp))
    
    results.append({
        'name': run_name,
        'program': program_str,
        'mse': mse,
        'length': est_gp_tuned._program.length_,
        'fitness': est_gp_tuned._program.raw_fitness_
    })
    
    print(f"Best symbolic function: {program_str}")
    print(f"MSE: {mse:.4f}")
    print(f"Length: {est_gp_tuned._program.length_}")
    print(f"Raw Fitness (MAE): {est_gp_tuned._program.raw_fitness_:.4f}")

    # Plotting
    y_pred_gp_tuned = est_gp_tuned.predict(X_train_gp)
    plt.figure(figsize=(10, 6))
    plt.scatter(x_data, y_data, label='Sample Data', color='blue', s=10, alpha=0.6)
    plt.plot(x_data, target_function(x_data), label='True Function', color='red', linestyle='--')
    plt.plot(x_data, y_pred_gp_tuned, label=f'gplearn ({run_name})', color='green', linestyle='-.')
    plt.title(f'gplearn Tuning: {run_name}')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend(loc='best', fontsize='small')
    plt.grid(True)
    plt.show()

print("\n--- Summary of Results ---")
for res in results:
    print(f"Run: {res['name']}, MSE: {res['mse']:.4f}, Length: {res['length']}, Fitness (MAE): {res['fitness']:.4f}, Function: {res['program']}")


### Observations from Parameter Tuning:

*   **Baseline:** (Comment on its performance)
*   **Larger Population & Generations:** (Comment on runtime, complexity, and accuracy changes)
*   **Higher Parsimony:** (Comment on function simplicity and accuracy)
*   **Lower Parsimony:** (Comment on function complexity and accuracy)
*   General thoughts on which parameters seemed most impactful.

## Applying gplearn to a Different 1D Dataset

Let's try `gplearn` on a new, perhaps more complex, 1D function to see how it performs. We'll define a new target function, generate data, and then attempt to find a symbolic expression for it.

In [ ]:
# Define a new target function
def target_function_new(x):
    return x**3 * 0.1 - np.cos(x * 2) + x * 0.3 # Example: a bit more complex

# Generate new data
x_data_new = np.linspace(-4, 4, 150) # Different range and number of points
y_data_new_true = target_function_new(x_data_new)
y_data_new_noisy = y_data_new_true + np.random.normal(0, 0.25, x_data_new.shape) # Slightly more noise

# Plot the new dataset
plt.figure(figsize=(10, 6))
plt.scatter(x_data_new, y_data_new_noisy, label='New Sample Data (Noisy)', color='purple', s=10, alpha=0.7)
plt.plot(x_data_new, y_data_new_true, label='New True Function', color='orange', linestyle='--')
plt.title('New 1D Dataset for gplearn')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

# Prepare data for gplearn
X_train_new_gp = x_data_new.reshape(-1, 1)

# Initialize SymbolicRegressor (using a decent parameter set from before)
# These are basic parameters; they can be tuned for better performance or faster convergence
# Re-using common_params from the previous cell if the notebook is run sequentially
# If not, redefine common_params here. For safety, let's redefine a suitable set.
params_for_new_dataset = {
    'population_size': 1500, # Adjusted population
    'generations': 25,       # Adjusted generations
    'stopping_criteria': 0.01,
    'p_crossover': 0.7,
    'p_subtree_mutation': 0.1,
    'p_hoist_mutation': 0.05,
    'p_point_mutation': 0.1,
    'max_samples': 0.9,
    'verbose': 1,
    'function_set': ('add', 'sub', 'mul', 'div', 'sin', 'cos', 'log', 'sqrt', 'neg', 'inv'), # inv can be useful
    'metric': 'mean absolute error',
    'parsimony_coefficient': 0.01, # Start with a moderate parsimony
    'random_state': 123, # Different random state for this experiment
    'n_jobs': -1
}

print("\n--- Applying gplearn to New Dataset ---")
print(f"Using parameters: {params_for_new_dataset}") # Could be more selective in printing if too verbose

est_gp_new = SymbolicRegressor(**params_for_new_dataset)

print("Fitting regressor to new data...")
est_gp_new.fit(X_train_new_gp, y_data_new_noisy)
print("Fitting complete.")

program_str_new = str(est_gp_new._program)
mse_new = mean_squared_error(y_data_new_noisy, est_gp_new.predict(X_train_new_gp))

print(f"Best symbolic function for new data: {program_str_new}")
print(f"MSE on new data: {mse_new:.4f}")
print(f"Length: {est_gp_new._program.length_}")
print(f"Raw Fitness (MAE): {est_gp_new._program.raw_fitness_:.4f}")

# Plotting results for the new dataset
y_pred_gp_new = est_gp_new.predict(X_train_new_gp)
plt.figure(figsize=(12, 7))
plt.scatter(x_data_new, y_data_new_noisy, label='New Sample Data', color='purple', s=10, alpha=0.6)
plt.plot(x_data_new, y_data_new_true, label='New True Function', color='orange', linestyle='--')
plt.plot(x_data_new, y_pred_gp_new, label=f'gplearn Prediction (New Data): {program_str_new}', color='teal', linestyle='-.')
plt.title('gplearn on New 1D Dataset')
plt.xlabel('x')
plt.ylabel('y')
plt.legend(loc='best', fontsize='small')
plt.grid(True)
plt.show()


### Observations for gplearn on the New Dataset:

*   How well did `gplearn` capture the new function?
*   Was the resulting symbolic function complex or simple?
*   Did it require more generations/population or different parameters to get a good fit compared to the first function?
*   Any functions from the `function_set` that were particularly useful or problematic?